# Análise Exploratória — Brazilian E-Commerce Public Dataset by Olist

**Tech Challenge — Fase 1 — Postech IA para Devs**
**Case:** Olist Intelligent Marketplace

Este notebook documenta, passo a passo, a análise exploratória usada como base para o
relatório executivo e o Mapa de Agentes de IA propostos para a Olist. Os dados são
reais, baixados diretamente do [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
(99.441 pedidos, 2016–2018).


## 1. Setup e carregamento dos dados

Carregamos as tabelas de pedidos, itens, pagamentos, avaliações, clientes, produtos e vendedores.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

orders = pd.read_csv("../data/olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"
])
items = pd.read_csv("../data/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
cat_translation = pd.read_csv("../data/product_category_name_translation.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## 2. Volumetria geral

Quantos pedidos, clientes, vendedores e produtos existem na base, e qual o período coberto.

In [2]:
print("Pedidos:", orders["order_id"].nunique())
print("Clientes únicos:", customers["customer_unique_id"].nunique())
print("Vendedores:", sellers["seller_id"].nunique())
print("Produtos:", products["product_id"].nunique())
print("Período:", orders["order_purchase_timestamp"].min(), "a", orders["order_purchase_timestamp"].max())
print()
print("Distribuição de status do pedido:")
orders["order_status"].value_counts()

Pedidos: 99441
Clientes únicos: 96096
Vendedores: 3095
Produtos: 32951
Período: 2016-09-04 21:15:19 a 2018-10-17 17:30:18

Distribuição de status do pedido:


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 3. Prazo de entrega e atraso

Calculamos o tempo de entrega (dias entre compra e entrega) e o atraso (entrega real vs. estimada), apenas para pedidos com status `delivered`.

In [3]:
delivered = orders[orders["order_status"] == "delivered"].copy()
delivered["delivery_days"] = (delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days
delivered["delay_days"] = (delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]).dt.days
delivered["is_late"] = delivered["delay_days"] > 0

print(f"Prazo médio de entrega: {delivered['delivery_days'].mean():.1f} dias (mediana {delivered['delivery_days'].median():.0f})")
print(f"Pedidos atrasados: {100*delivered['is_late'].mean():.2f}%")
print(f"Atraso médio quando atrasado: {delivered.loc[delivered['is_late'], 'delay_days'].mean():.1f} dias")

Prazo médio de entrega: 12.1 dias (mediana 10)
Pedidos atrasados: 6.77%
Atraso médio quando atrasado: 10.6 dias


## 4. Reviews: nota média e efeito do atraso

Este é um dos achados mais importantes: cruzamos pontualidade da entrega com a nota dada pelo cliente.

In [4]:
print(f"Nota média geral: {reviews['review_score'].mean():.2f}")
print(f"Avaliações 1-2 estrelas: {100*reviews['review_score'].isin([1,2]).mean():.2f}%")
print(f"Avaliações com comentário escrito: {100*reviews['review_comment_message'].notna().mean():.2f}%")
print()

rev_orders = delivered.merge(reviews[["order_id", "review_score"]], on="order_id", how="inner")
score_by_lateness = rev_orders.groupby("is_late")["review_score"].mean().round(2)
print("Nota média — pedido no prazo:", score_by_lateness.get(False))
print("Nota média — pedido atrasado:", score_by_lateness.get(True))

Nota média geral: 4.07
Avaliações 1-2 estrelas: 15.09%
Avaliações com comentário escrito: 41.75%

Nota média — pedido no prazo: 4.28
Nota média — pedido atrasado: 2.26


## 5. Taxa de recompra

O achado mais impactante do diagnóstico. Usamos `customer_unique_id` (identifica a pessoa) em vez de `customer_id` (identifica o registro do pedido, que muda a cada compra).

In [5]:
orders_cust = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
purchase_counts = orders_cust.groupby("customer_unique_id")["order_id"].nunique()

print(f"Clientes com apenas 1 compra: {100*(purchase_counts == 1).mean():.2f}%")
print(f"Clientes com recompra (2+): {100*(purchase_counts > 1).mean():.2f}%")
print(f"Média de pedidos por cliente: {purchase_counts.mean():.3f}")

Clientes com apenas 1 compra: 96.88%
Clientes com recompra (2+): 3.12%
Média de pedidos por cliente: 1.035


## 6. Pagamentos e catálogo de produtos

In [6]:
print(f"Ticket médio: R$ {payments['payment_value'].mean():.2f}")
print(f"Parcelamento médio: {payments['payment_installments'].mean():.2f}x")
print()
print("Formas de pagamento:")
print(payments["payment_type"].value_counts())

Ticket médio: R$ 154.10
Parcelamento médio: 2.85x

Formas de pagamento:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [7]:
cat_map = dict(zip(cat_translation["product_category_name"], cat_translation["product_category_name_english"]))
top_cats = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
top5 = top_cats["product_category_name"].value_counts().head(5)
top5.index = [cat_map.get(c, c) for c in top5.index]
print(f"Total de categorias: {products['product_category_name'].nunique()}")
print("\nTop 5 categorias por volume de itens vendidos:")
top5

Total de categorias: 73

Top 5 categorias por volume de itens vendidos:


bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
Name: count, dtype: int64

## 7. Distribuição geográfica e atraso por estado

Cruzamos o estado do cliente com prazo médio de entrega e taxa de atraso — usado para priorizar o monitoramento logístico por região no Agente de Logística.

In [8]:
top_states = customers["customer_state"].value_counts().head(5)
print("Top 5 estados por volume de pedidos:")
print(top_states)
print()

d_state = delivered.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
late_by_state = d_state.groupby("customer_state")["is_late"].mean().mul(100).round(2)
delivery_by_state = d_state.groupby("customer_state")["delivery_days"].mean().round(1)

import pandas as pd
resumo = pd.DataFrame({
    "pedidos": top_states,
    "atraso_%": [late_by_state.get(s) for s in top_states.index],
    "prazo_medio_dias": [delivery_by_state.get(s) for s in top_states.index],
})
resumo

Top 5 estados por volume de pedidos:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64



,pedidos,atraso_%,prazo_medio_dias
customer_state,,,
SP,41746,4.49,8.3
RJ,12852,12.11,14.8
MG,11635,4.57,11.5
RS,5466,6.08,14.8
PR,5045,4.04,11.5


## 8. Síntese dos achados

| Dimensão | Achado |
|---|---|
| Prazo de entrega | ~12,1 dias em média; 6,8% dos pedidos atrasam |
| Nota vs. atraso | Cai de 4,28 (no prazo) para 2,26 (atrasado) |
| Reviews | Nota média 4,07; 15,1% são 1-2 estrelas; 41,8% têm comentário |
| Recompra | Apenas 3,1% dos clientes compram mais de uma vez |
| Geografia | RJ tem o maior atraso (12,1%) e maior prazo médio (14,8 dias) entre os 5 maiores estados |

Esses achados embasam diretamente o **Mapa de Agentes de IA** proposto no relatório executivo:
Agente de Atendimento, Agente de Reviews, Agente de Logística e Agente de Churn — cada um
endereçando uma das oportunidades identificadas aqui.

Repositório e relatório completo: ver `README.md`.
